# 10年定着予測 - 基盤モデル3種 × 特徴量セットの検証（68_）

`67_` に **TabDPT** を追加したもの。`67_` の代わりにこちらを実行する。

## 問題意識

現在の最良（Public 0.508793）のTabPFNは **CatBoost重要度の上位150列**で学習している。
しかしこれは**CatBoostの帰納バイアスで選んだ特徴量**であり、
Transformerのin-context learningに最適である保証はない。

`63_`が示したのは「441列→150列で改善(0.0145)」であって、
**「CatBoost重要度で選ぶのが最良」ではない**。同じ150列でも選び方で変わりうる。

## 検証する基盤モデル3種（機構がそれぞれ異なる）

| モデル | 事前学習 | 推論機構 | ライセンス |
|---|---|---|---|
| **TabPFN v2** | 合成データのみ | 全文脈をattention | Prior Labs License v1.1（Apache 2.0派生・商用可）|
| **TabICL** | 合成データ | **列→行の2段階attention** | BSD 3-Clause（商用可）|
| **TabDPT** | **実データ123セット** | **faissでkNN検索して文脈を選ぶ（retrieval型）** | Apache-2.0（商用可）|

**TabDPTは事前学習データも推論機構も他2つと本質的に違う**ため、
アンサンブル相手として最も質の高い多様性が期待できる。

## 検証する特徴量セット

| 候補 | 選び方 | 仮説 |
|---|---|---|
| `cb_topK` (K=50/100/150/200) | CatBoost重要度上位 | 現行方式・ベースライン |
| `mi_top150` | 相互情報量（モデル非依存）| CatBoostのバイアスを外す |
| `decorr150` | 相関プルーニング後に重要度上位 | 木は冗長列に鈍感だが、Transformerは全列にattendするので冗長性が希釈要因になりうる |
| **`hire_fixed`** | persona + L2 + deptte + derived + tfidf（**月次集約を全落とし**）| [[monthly-data-information-ceiling]]で「月次データに10年定着の情報がほぼ無い」と確定済み。**列数制約の厳しい基盤モデルほど、約400列の月次列を落とす効果が大きいはず** |
| `full441` | 全部 | `63_`の劣化を再現する対照 |

## 評価の仕方

**単体valだけでなくCatBoostとのブレンド改善も見る。** 実運用ではブレンドして使うので、
「単体は劣るが多様性が高くブレンドは良い」構成がありうるため。

**どれか1つを検証スコアで選ぶことはしない。** `63_`の0.0145のような大差だけ採用し、
それ以下は[[ablation-cannot-settle-feature-blocks]]の勝者の呪いとして複数平均で使う。

## 実行環境: Colab CPUハイメモリ

ローカル実測（M系CPU、2208学習→535予測、1シードあたり）:

| 設定 | 時間 | ピークRSS |
|---|---|---|
| TabICL 既定（n_est=8, **batch_size=8**）| 53.1秒 | **10.98 GB** ← 標準RAMだと落ちる |
| **TabICL n_est=8, batch_size=1** | **29.2秒** | **4.98 GB** |
| **TabPFN n_est=4, memory_saving_mode=True** | 123.9秒 | **2.75 GB** |
| TabPFN n_est=4, memory_saving なし | 151.4秒 | 2.84 GB |

**省メモリ設定は遅くなるどころか速い**（メモリ逼迫によるスワップが解消されるため）。
本ノートブックは所要時間とピークRSSを毎回表示し、モデルは1つずつロードして即解放する。

## ライセンス表記（Prior Labs License 第10条）

> **Built with PriorLabs-TabPFN**

使用する3モデルはいずれも**商用利用可**で、SIGNATE参加規約 第2条8項に適合する。
詳細は `SUBMISSION_README.md` 第6節を参照。


In [41]:
# tabpfn はバージョン固定必須。
# 2.5/2.6/3系(PyPIの6.x〜8.x)は**非商用ライセンス**で、SIGNATE参加規約 第2条8項
# （商業利用が禁止されているOSSの利用・組込み禁止）に抵触するため使用不可。
# 2.x系 = TabPFN v2 = Prior Labs License v1.1（Apache 2.0派生・商用可）。
!pip install -q catboost optuna "tabpfn==2.2.1" tabicl tabdpt

# 「ERROR: pip's dependency resolver...」は依存解決の警告であって失敗ではない。
# 既に別バージョンのtabpfnが入っている場合は、このセルの後で
# 「ランタイム → セッションを再起動」してから最初のセルに戻ること。


In [42]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [43]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [45]:
SCRIPT_NAME = "68_foundation_models_3way"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = False  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-16 06:16:52] [INFO] === [68_foundation_models_3way] 実験開始 ===


INFO:68_foundation_models_3way:=== [68_foundation_models_3way] 実験開始 ===


[2026-08-16 06:16:52] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


INFO:68_foundation_models_3way:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


[2026-08-16 06:16:52] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/68_foundation_models_3way_checkpoint.csv


INFO:68_foundation_models_3way:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/68_foundation_models_3way_checkpoint.csv


[2026-08-16 06:16:52] [INFO] チェックポイントは未作成（新規実行）


INFO:68_foundation_models_3way:チェックポイントは未作成（新規実行）


In [46]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-16 06:16:53] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:68_foundation_models_3way:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-16 06:16:53] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:68_foundation_models_3way:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-16 06:16:53] [INFO] 定着率: 0.5647


INFO:68_foundation_models_3way:定着率: 0.5647


[2026-08-16 06:16:53] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:68_foundation_models_3way:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [47]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-16 06:16:53] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:68_foundation_models_3way:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-16 06:16:53] [INFO] Test  早期退職者: 0名 / 2502名


INFO:68_foundation_models_3way:Test  早期退職者: 0名 / 2502名


[2026-08-16 06:16:53] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:68_foundation_models_3way:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-16 06:16:53] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:68_foundation_models_3way:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`51_`と同一ロジック）

In [48]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [49]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-16 06:16:53] [INFO] ------------------------------------------------------------


INFO:68_foundation_models_3way:------------------------------------------------------------


[2026-08-16 06:16:53] [INFO] split非依存の基本特徴量を生成中...


INFO:68_foundation_models_3way:split非依存の基本特徴量を生成中...


[2026-08-16 06:16:53] [INFO] ------------------------------------------------------------


INFO:68_foundation_models_3way:------------------------------------------------------------


[2026-08-16 06:22:19] [INFO] split非依存の基本特徴量生成完了


INFO:68_foundation_models_3way:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`51_`と同一・継続採用）

In [50]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-16 06:22:20] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:68_foundation_models_3way:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-16 06:22:21] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:68_foundation_models_3way:入社時メモ: SVD累積寄与率=0.760


[2026-08-16 06:22:24] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:68_foundation_models_3way:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-16 06:22:26] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:68_foundation_models_3way:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-16 06:22:26] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:68_foundation_models_3way:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`51_`と同一・継続採用）

In [51]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-16 06:22:26] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:68_foundation_models_3way:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-16 06:24:25] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:68_foundation_models_3way:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`51_`と同一）

In [52]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-16 06:24:25] [INFO] Persona単位の基本特徴量を生成中...


INFO:68_foundation_models_3way:Persona単位の基本特徴量を生成中...


[2026-08-16 06:24:25] [INFO] Persona単位の基本特徴量処理完了


INFO:68_foundation_models_3way:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版、`51_`と同一）

`51_`と同じくv1/v2両方を生成するが、実際に特徴量として使うのはv2（`BLOCK={"L2"}`）のみ
（`28_`以降ずっとv2が現在の最良）。


In [53]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 50_: 見出しがない書式B（276件、5.24%）のフォールバック（49_で確認済み・Public -0.0022〜-0.0035）。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-16 06:24:25] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:68_foundation_models_3way:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-16 06:24:25] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:68_foundation_models_3way:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-16 06:24:25] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:68_foundation_models_3way:L_v2: Train (2761, 3), Test (2502, 3)


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数（`51_`と同一）

In [54]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（51_と完全に同一ロジック）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")


✅ 部署Target Encoding・prepare_split関数定義完了


## 7. 特徴量の組み立て（`51_`と同一、`BLOCK={"L2"}`固定）

In [55]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-16 06:24:26] [INFO] ============================================================


INFO:68_foundation_models_3way:============================================================


[2026-08-16 06:24:26] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:68_foundation_models_3way:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-16 06:24:26] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:68_foundation_models_3way:main_train=2208, main_valid(生存者)=535


[2026-08-16 06:24:26] [INFO] 全件=2761


INFO:68_foundation_models_3way:全件=2761


[2026-08-16 06:24:26] [INFO] 特徴量数: 441


INFO:68_foundation_models_3way:特徴量数: 441


## 8. 特徴量グループの棚卸し（`51_`から移植、内容は同一）

In [56]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


## 9. 設定

In [57]:
import gc, resource, time, warnings
import torch, tabpfn
from tabpfn import TabPFNClassifier
from tabicl import TabICLClassifier
from tabdpt import TabDPTClassifier
from importlib.metadata import version as _pkgver
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings("ignore")

_disk, _live = _pkgver("tabpfn"), tabpfn.__version__
if not _live.startswith("2."):
    raise RuntimeError(
        f"tabpfn: ディスク {_disk} / 実行中 {_live}。v2系(2.x)が必要。\n"
        "（2.5/3系は非商用ライセンスで参加規約 第2条8項に抵触するため使用不可）\n"
        + ("→ ランタイム → セッションを再起動 してください。" if _disk.startswith("2.")
           else "→ セル1を実行し、その後再起動してください。"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
peak_gb = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1 << 30)
print(f"tabpfn {_live} / tabicl {_pkgver('tabicl')} / tabdpt {_pkgver('tabdpt')}")
print(f"device = {DEVICE} / 現在のピークRSS {peak_gb():.2f} GB")

# --- 調整パラメータ ---
SEEDS_FM = [42, 2024, 7]
MODELS = ["tabdpt", "tabicl", "tabpfn"]   # 速い順。時間が足りなければ後ろを削る

# ローカル実測（M系CPU, 2208学習→535予測, 1シード）:
#   TabDPT              :   7.5秒 / 1.39GB  ← fit 6.5s + predict 1.0s。seedはpredict時に指定
#   TabICL (batch_size=1):  29.2秒 / 4.98GB  ← 既定batch_size=8だと53.1秒/10.98GBで標準RAMだと落ちる
#   TabPFN (n_est=4,省メモリ): 123.9秒 / 2.75GB
TABICL_KW = dict(n_estimators=8, batch_size=1)
TABPFN_KW = dict(n_estimators=4, memory_saving_mode=True)
# ⚠️ 現最良(0.508793)を出した 63_ は TabPFN を既定(n_estimators=8)で使っている。
#    ここで4にしているのは検証を速く回すため。全構成で同一設定なので**セット間の比較は公平**だが、
#    採用が決まってTest予測を作り直す際は 8 に戻すこと。

NOISE_HINT = 0.003        # 単一シードsdの目安。これ以下の差は読まない
W_BLEND = 0.7             # CatBoost側の重み。63_で決めた値。ここでは走査しない

A_PARAMS = {"depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
            "border_count": 218, "bagging_temperature": 0.6787467566574921,
            "random_strength": 1.438494697238285}
ITER_FIXED = 560
SEEDS_CB = [42, 2024, 7, 1234, 99]

FEATS_ALL = _feature_cols(ag_train_80b)
y_val = ag_val_surv[TARGET_COL].values
assert len(FEATS_ALL) == 441, f"{len(FEATS_ALL)}列（441列のはず）"
print(f"\n全特徴量 {len(FEATS_ALL)} 列 / 学習 {len(ag_train_80b)}件 / 検証 {len(ag_val_surv)}件")
print(f"モデル: {MODELS} / シード: {SEEDS_FM}")


tabpfn 2.2.1 / tabicl 2.1.1 / tabdpt 1.2.0
device = cpu / 現在のピークRSS 0.00 GB

全特徴量 441 列 / 学習 2208件 / 検証 535件
モデル: ['tabdpt', 'tabicl', 'tabpfn'] / シード: [42, 2024, 7]


## 10. CatBoost（ベースライン兼、重要度の供給源）

In [58]:
def cb_run(feats, seeds=SEEDS_CB):
    obj = [c for c in feats if ag_train_80b[c].dtype == "object"]
    Xtr, ytr = ag_train_80b[feats].fillna(-999), ag_train_80b[TARGET_COL]
    Xva = ag_val_surv[feats].fillna(-999)
    ps, ms = [], []
    for s in seeds:
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER_FIXED, random_seed=s, verbose=False,
                                  cat_features=obj, task_type="CPU")
        m.fit(Xtr, ytr); ms.append(m); ps.append(m.predict_proba(Xva)[:, 1])
    return np.mean(ps, axis=0), ms


t0 = time.time()
cb_val_full, cb_models = cb_run(FEATS_ALL)
CB_VAL = log_loss(y_val, cb_val_full)
print(f"CatBoost(441列) val = {CB_VAL:.6f}  ({time.time()-t0:.0f}秒)")

imp = pd.Series(np.mean([m.get_feature_importance() for m in cb_models], axis=0),
                index=FEATS_ALL).sort_values(ascending=False)
imp.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_catboost_importance.csv")


CatBoost(441列) val = 0.516147  (22秒)


## 11. 候補となる特徴量セット（すべて学習データのみから作る）

In [59]:
def numeric_matrix(feats, df):
    """相関/相互情報量の計算用に数値化する。

    注意: 単純な fillna(median) では**全行NaNの列**（中央値もNaNになる）が埋まらず、
    sklearn が ValueError("Input X contains NaN") で落ちる。
    さらに np.corrcoef が NaN を返し、decorr の判定（|r|<=0.95）が常に偽になって
    列がほとんど採用されない、という静かな不具合にもなる。
    そのため inf の除去 → 中央値 → 0 の順で確実に埋める。
    """
    M = df[feats].copy()
    for c in feats:
        if M[c].dtype == "object":
            M[c] = M[c].astype("category").cat.codes.astype(float)
    M = M.astype(np.float64).replace([np.inf, -np.inf], np.nan)
    n_allnan = int(M.isna().all().sum())
    M = M.fillna(M.median()).fillna(0.0)          # 全行NaNの列は0で埋める
    assert not M.isna().any().any(), "NaNが残っている"
    assert np.isfinite(M.values).all(), "infが残っている"
    if n_allnan:
        print(f"  （全行NaNの列 {n_allnan} 本を0で補填した）")
    return M.astype(np.float32)


FEATURE_SETS = {}
for K in [50, 100, 150, 200]:
    FEATURE_SETS[f"cb_top{K}"] = imp.head(K).index.tolist()

t0 = time.time()
Mtr = numeric_matrix(FEATS_ALL, ag_train_80b)
mi = pd.Series(mutual_info_classif(Mtr, ag_train_80b[TARGET_COL], random_state=42),
               index=FEATS_ALL).sort_values(ascending=False)
FEATURE_SETS["mi_top150"] = mi.head(150).index.tolist()
mi.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_mutual_info.csv")
print(f"相互情報量（{time.time()-t0:.0f}秒） 上位5: {mi.head(5).index.tolist()}")

t0 = time.time()
corr = np.corrcoef(Mtr.values, rowvar=False); np.fill_diagonal(corr, 0.0)
kept = []
for c in imp.index:                      # 重要度順に見て、既採用と|r|>0.95なら捨てる
    i = FEATS_ALL.index(c)
    if all(abs(corr[i, j]) <= 0.95 for j in kept):
        kept.append(i)
    if len(kept) >= 150:
        break
FEATURE_SETS["decorr150"] = [FEATS_ALL[i] for i in kept]
print(f"相関プルーニング（{time.time()-t0:.0f}秒）: {len(kept)}列")

HIRE_GROUPS = ["persona", "deptte", "derived", "L2", "tfidf"]
hire_cols = {c for g in HIRE_GROUPS if g in FEATURE_GROUPS for c in FEATURE_GROUPS[g]}
FEATURE_SETS["hire_fixed"] = [c for c in FEATS_ALL if c in hire_cols]
FEATURE_SETS["full441"] = FEATS_ALL

print()
for k, v in FEATURE_SETS.items():
    print(f"  {k:12s} {len(v):4d}列")
base = set(FEATURE_SETS["cb_top150"])
print("\ncb_top150 との重複率（選び方が実際に違うかの確認）:")
for k in ["mi_top150", "decorr150", "hire_fixed"]:
    print(f"  {k:12s} {len(base & set(FEATURE_SETS[k]))/len(base)*100:5.1f}%")


  （全行NaNの列 2 本を0で補填した）
相互情報量（3秒） 上位5: ['残業時間_mean', 'cluster', '残業時間_cv', '残業時間_median', '残業時間_min']
相関プルーニング（0秒）: 150列

  cb_top50       50列
  cb_top100     100列
  cb_top150     150列
  cb_top200     200列
  mi_top150     150列
  decorr150     150列
  hire_fixed     78列
  full441       441列

cb_top150 との重複率（選び方が実際に違うかの確認）:
  mi_top150     43.3%
  decorr150     91.3%
  hire_fixed    31.3%


## 12. 基盤モデル3種での評価

**モデルは1つずつロードして使い終わったら解放**する（メモリ逼迫を避けるため）。
TabDPTは`seed`が`predict_proba`の引数なので、**fitは1回だけで各シードの予測を取る**。


In [60]:
def to_matrix(feats):
    """基盤モデル用。NaNは基本的に埋めない（3モデルともネイティブに扱える）。
    カテゴリは序数コード化。

    例外: **全行NaNの列**だけは0で埋める。TabICLは全行NaNの列があると
    内部の重複列検出(unique_filter_)がクラッシュする
    （IndexError: boolean index did not match indexed array...、ローカルで再現確認済み）。
    部分欠損の列はネイティブなNaN処理に任せる（そちらは問題なく動作する）。
    """
    obj = [c for c in feats if ag_train_80b[c].dtype == "object"]
    both = pd.concat([ag_train_80b[feats], ag_val_surv[feats]])
    cmap = {c: {v: i for i, v in enumerate(sorted(both[c].astype(str).unique()))} for c in obj}

    num_cols = [c for c in feats if c not in obj]
    allnan_cols = [c for c in num_cols if ag_train_80b[c].isna().all()]
    if allnan_cols:
        print(f"  （全行NaNの列 {len(allnan_cols)} 本を0で補填: {allnan_cols[:5]}{'...' if len(allnan_cols) > 5 else ''}）")

    out = []
    for df in (ag_train_80b, ag_val_surv):
        M = df[feats].copy()
        for c in obj:
            M[c] = df[c].astype(str).map(cmap[c]).astype(float)
        for c in allnan_cols:
            M[c] = M[c].fillna(0.0)
        out.append(M.astype(np.float32).values)
    return out[0], out[1], [feats.index(c) for c in obj]


def fm_eval(kind, feats, seeds=SEEDS_FM):
    """基盤モデルの検証予測（シード平均）と所要時間を返す。3モデルのAPI差を吸収する。"""
    Mtr, Mva, cat_idx = to_matrix(feats)
    ytr = ag_train_80b[TARGET_COL].values
    t0, ps = time.time(), []

    if kind == "tabdpt":
        # seedはpredict_probaの引数。fitは1回でよい（fit 6.5s / predict 1.0s）
        clf = TabDPTClassifier(device=DEVICE)
        clf.fit(Mtr, ytr)
        for s in seeds:
            ps.append(clf.predict_proba(Mva, seed=s)[:, 1])
        del clf
    elif kind == "tabicl":
        for s in seeds:
            clf = TabICLClassifier(device=DEVICE, random_state=s, **TABICL_KW)
            clf.fit(Mtr, ytr); ps.append(clf.predict_proba(Mva)[:, 1]); del clf
    else:  # tabpfn
        for s in seeds:
            for extra in ({"categorical_features_indices": cat_idx,
                           "ignore_pretraining_limits": True, **TABPFN_KW},
                          {"categorical_features_indices": cat_idx, **TABPFN_KW},
                          {"categorical_features_indices": cat_idx}, {}):
                try:
                    clf = TabPFNClassifier(device=DEVICE, random_state=s, **extra); break
                except TypeError:
                    continue
            clf.fit(Mtr, ytr); ps.append(clf.predict_proba(Mva)[:, 1]); del clf
    gc.collect()
    return np.mean(ps, axis=0), np.array(ps), time.time() - t0


results, val_preds = [], {}
for name, feats in FEATURE_SETS.items():
    cbv, _ = cb_run(feats, seeds=SEEDS_CB[:3])
    row = {"構成": name, "列数": len(feats), "CatBoost": log_loss(y_val, cbv)}
    for kind in MODELS:
        pv, allp, el = fm_eval(kind, feats)
        val_preds[(name, kind)] = pv
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{kind}_{name}_valpreds.npy", pv)
        row[kind] = log_loss(y_val, pv)
        row[f"bl_{kind}"] = log_loss(y_val, np.clip(W_BLEND*cb_val_full + (1-W_BLEND)*pv, 1e-9, 1-1e-9))
        row[f"{kind}_秒"] = round(el, 1)
        seed_mad = float(np.mean([np.abs(allp[i]-allp[j]).mean()
                                  for i in range(len(allp)) for j in range(i+1, len(allp))])) \
                   if len(allp) > 1 else np.nan
        row[f"{kind}_seedMAD"] = round(seed_mad, 4)
        print(f"  [{name:12s} {kind:7s}] val={row[kind]:.6f} blend={row[f'bl_{kind}']:.6f} "
              f"seedMAD={seed_mad:.4f}  {el:5.0f}秒  RSS {peak_gb():.2f}GB")
    results.append(row); gc.collect()

R = pd.DataFrame(results)
pd.set_option("display.width", 260)
print()
print(R.round(6).to_string(index=False))
R.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_results.csv", index=False)


  [cb_top50     tabdpt ] val=0.564243 blend=0.524545 seedMAD=0.0408     10秒  RSS 0.00GB
  [cb_top50     tabicl ] val=0.543874 blend=0.517933 seedMAD=0.0207    110秒  RSS 0.00GB
  [cb_top50     tabpfn ] val=0.507722 blend=0.506699 seedMAD=0.0207    272秒  RSS 0.00GB
  [cb_top100    tabdpt ] val=0.544817 blend=0.520406 seedMAD=0.0450     10秒  RSS 0.00GB
  [cb_top100    tabicl ] val=0.541308 blend=0.518873 seedMAD=0.0143    185秒  RSS 0.00GB
  [cb_top100    tabpfn ] val=0.494140 blend=0.504975 seedMAD=0.0208    544秒  RSS 0.00GB
  [cb_top150    tabdpt ] val=0.547778 blend=0.519221 seedMAD=0.0960     10秒  RSS 0.00GB
  [cb_top150    tabicl ] val=0.542665 blend=0.519524 seedMAD=0.0143    266秒  RSS 0.00GB
  [cb_top150    tabpfn ] val=0.503060 blend=0.508486 seedMAD=0.0223    866秒  RSS 0.00GB
  [cb_top200    tabdpt ] val=0.546537 blend=0.518615 seedMAD=0.0841     10秒  RSS 0.00GB
  [cb_top200    tabicl ] val=0.541811 blend=0.519652 seedMAD=0.0151    344秒  RSS 0.01GB
  [cb_top200    tabpfn ] val=0.5

## 13. 判定

In [61]:
print(f"CatBoost(441列)単体 val = {CB_VAL:.6f}")
print(f"読み取りの下限（単一シードsd目安）= {NOISE_HINT}\n")

for kind in MODELS:
    if kind not in R.columns: continue
    sub = R[["構成", "列数", kind, f"bl_{kind}", f"{kind}_seedMAD"]].copy()
    bb = sub[f"bl_{kind}"].min()
    sub["ブレンド_最良差"] = sub[f"bl_{kind}"] - bb
    sub["判定"] = np.where(sub["ブレンド_最良差"] <= NOISE_HINT, "最良と区別できない", "劣る")
    print(f"=== {kind} ===")
    print(sub.round(6).to_string(index=False))
    print(f"  → 最良と区別できない: {sub.loc[sub['判定']=='最良と区別できない','構成'].tolist()}\n")

# 3モデル間の多様性（同一特徴量セットでの相関）
print("=" * 78)
print("同一特徴量セットにおけるモデル間の予測相関（低いほどアンサンブル価値が高い）")
for name in FEATURE_SETS:
    got = [(k, val_preds[(name, k)]) for k in MODELS if (name, k) in val_preds]
    if len(got) < 2: continue
    line = f"  {name:12s} "
    for i in range(len(got)):
        for j in range(i+1, len(got)):
            line += f"{got[i][0][:4]}-{got[j][0][:4]}={np.corrcoef(got[i][1],got[j][1])[0,1]:.3f}  "
    line += f"| vs CatBoost: " + "  ".join(
        f"{k[:4]}={np.corrcoef(cb_val_full,v)[0,1]:.3f}" for k, v in got)
    print(line)

# 3モデル平均 + CatBoost のブレンド（参考）
print()
print("=" * 78)
print("参考: 基盤モデル3種を平均してからCatBoostとブレンドした場合")
for name in FEATURE_SETS:
    got = [val_preds[(name, k)] for k in MODELS if (name, k) in val_preds]
    if len(got) < 2: continue
    fm = np.mean(got, axis=0)
    bl = log_loss(y_val, np.clip(W_BLEND*cb_val_full + (1-W_BLEND)*fm, 1e-9, 1-1e-9))
    print(f"  {name:12s} 3モデル平均単体={log_loss(y_val,fm):.6f}  CatBoostとブレンド={bl:.6f}"
          f"  (CatBoost単体比 {bl-CB_VAL:+.6f})")


CatBoost(441列)単体 val = 0.516147
読み取りの下限（単一シードsd目安）= 0.003

=== tabdpt ===
        構成  列数   tabdpt  bl_tabdpt  tabdpt_seedMAD  ブレンド_最良差        判定
  cb_top50  50 0.564243   0.524545          0.0408  0.005930        劣る
 cb_top100 100 0.544817   0.520406          0.0450  0.001791 最良と区別できない
 cb_top150 150 0.547778   0.519221          0.0960  0.000606 最良と区別できない
 cb_top200 200 0.546537   0.518615          0.0841  0.000000 最良と区別できない
 mi_top150 150 0.567502   0.524795          0.0766  0.006180        劣る
 decorr150 150 0.555306   0.521774          0.0777  0.003159        劣る
hire_fixed  78 0.573325   0.525382          0.0346  0.006767        劣る
   full441 441 0.578965   0.527485          0.0846  0.008870        劣る
  → 最良と区別できない: ['cb_top100', 'cb_top150', 'cb_top200']

=== tabicl ===
        構成  列数   tabicl  bl_tabicl  tabicl_seedMAD  ブレンド_最良差        判定
  cb_top50  50 0.543874   0.517933          0.0207  0.010630        劣る
 cb_top100 100 0.541308   0.518873          0.0143  0.011570        劣る
 cb

## 14. 次のアクション

- **`63_`の0.0145のような大差**が出た構成があれば採用し、Test予測を作り直す
- **差が`NOISE_HINT`以下なら複数構成の平均を使う**（[[ablation-cannot-settle-feature-blocks]]）
- **モデル間相関が低ければ3モデル平均をアンサンブルに使う**価値がある。
  TabDPTは事前学習データ（実データ）も推論機構（faissでkNN検索するretrieval型）も
  他2つと本質的に違うので、ここが最も期待できる
- `hire_fixed`（月次列を全落とし）が明確に良ければ、
  [[monthly-data-information-ceiling]]の結論が**基盤モデルでより強く効く**ことの実証になる

### このノートブックがやらないこと

- **ブレンド重みの走査**（w=0.7固定）。Publicでも検証でも重みを探索すると
  Private評価に対する過学習になる（[[private-lb-variance-strategy]]）
- **Test予測の生成**。検証専用。採用が決まってから別途作る
- 検証は TabPFN n_estimators=4 で回すが、**Test予測を作る際は`63_`と同じ既定(8)に戻す**
